# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khuld13/ML-intern-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item (a single article or page on a client's site), for one client, on one calendar day. This matches the grain of fact_content_daily_performance directly. "Content item" here means one published article or page — the warehouse doesn't distinguish blog posts from product pages at this level, it just tracks each one as a unit. For this notebook I develop and test on a single mid-panel month, month=2026-03, so I can iterate quickly without hitting Hugging Face rate limits or accidentally touching the sealed final month.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Table used: fact_content_daily_performance only — daily gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, joined implicitly by client_hash_id + content_hash_id + report_date (the table's own grain).
Features: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, and derived ctr — all observable signals logged after the fact, before any decision is made.
Label/proxy: is_declining_label = 1 if gsc_avg_position > 10, else 0. This is a simple page-one-vs-not proxy built directly for this notebook's leakage demonstration in Section 3. It replaces the starter repo's trend_direction-based label from notebook 02 — I chose a position-based proxy here since it's directly computable from the columns actually available in this table, and it still lets me demonstrate the same leakage lesson (a label-derived feature causing an artificially perfect score).
Excluded: health_score and any other FlyRank product decision flag. These are the app's own rule-based outputs, not observable signals — using them as features would let a model just copy FlyRank's existing answer instead of learning anything new (a circular result). Also excluded: dim_content fields like word_count and content_age_days — not used in this notebook's verification queries, since Section 3 works entirely from the daily fact table.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')  # pulled from Colab Secrets, never printed or stored in the file

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
print("Connected.")

Connected.


In [4]:
q1 = con.sql("""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n_rows
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()
q1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,n_rows


In [5]:
print(len(q1))
q1

0


,client_hash_id,content_hash_id,report_date,n_rows


Grain confirmed: 0 duplicate (client, content, date) combinations found in month=2026-03. One row = one content item, for one client, on one day, as claimed in Section 1.

In [6]:
q2 = con.sql("""
    SELECT
        COUNT(*) AS n_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()
q2

,n_rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


March 2026 slice confirmed: 9,841,378 rows, spanning the full month (2026-03-01 to 2026-03-31) — matches the expected single-month window from the data contract

In [7]:
q3 = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_with_ga4,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_with_ga4
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()
q3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_ga4,pct_with_ga4
0,9841378,413966,4.2


Availability check: only 413,966 of 9,841,378 rows (4.2%) have GA4 engagement data available (ga4_data_available IS TRUE). The rest are search-only rows — consistent with the unbalanced-panel limitation noted in Section 4.

In [8]:
con.sql("""
    DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [9]:
features = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        ROUND(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0), 4) AS ctr
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    LIMIT 20
""").df()
features

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ctr
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,<NA>,0.0000
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,<NA>,0.0000
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,<NA>,0.0080
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,<NA>,0.0000
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,<NA>,0.0000
5,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,239,1,7.347280,<NA>,0.0042
6,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,191,0,7.832461,<NA>,0.0000
7,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,55,0,3.272727,<NA>,0.0000
8,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01,77,0,5.636364,<NA>,0.0000
9,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03-01,2,0,4.500000,<NA>,0.0000


Five features — why each is knowable at the decision moment:

* gsc_impressions — knowable because it's a completed, logged daily
search-console measurement; no future data is used.
* gsc_clicks — knowable because it's an observed daily count of clicks that already happened that day.
* gsc_avg_position — knowable because it reflects where the page actually ranked that day, not a forecast.
* ga4_sessions — knowable because it's a completed daily analytics count, recorded after sessions occurred.
* ctr (clicks ÷ impressions) — knowable because it's calculated purely from that same day's own gsc_clicks and gsc_impressions, with no outside-window information.

In [10]:
model_df = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ROUND(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0), 4) AS ctr,
        CASE WHEN gsc_avg_position > 10 THEN 1 ELSE 0 END AS is_declining_label
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE gsc_impressions > 0
""").df()

model_df = model_df.dropna()
print(model_df.shape)
model_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(3611061, 8)


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,is_declining_label
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,0.000,0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,0.000,0
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,0.008,0
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,0.000,0
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,0.000,0


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

X_honest = model_df[["gsc_impressions", "gsc_clicks", "ctr"]]
y = model_df["is_declining_label"]

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.2, random_state=42)

clf_honest = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_honest.fit(X_train, y_train)

honest_auc = roc_auc_score(y_test, clf_honest.predict_proba(X_test)[:, 1])
print("Honest AUC:", honest_auc)

Honest AUC: 0.5896601738966991


In [12]:
X_leaked = model_df[["gsc_impressions", "gsc_clicks", "ctr", "gsc_avg_position"]]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaked, y, test_size=0.2, random_state=42)

clf_leaked = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_leaked.fit(X_train_l, y_train_l)

leaked_auc = roc_auc_score(y_test_l, clf_leaked.predict_proba(X_test_l)[:, 1])
print("Leaked AUC:", leaked_auc)

Leaked AUC: 1.0


Leakage trap, performed: adding gsc_avg_position as a feature — the same column the label (is_declining_label = gsc_avg_position > 10) was derived from — pushed AUC from 0.590 (honest) to 1.0 (leaked). This is a textbook circular result: the model isn't learning any real relationship, it's just recovering the exact rule that built its own target. gsc_avg_position is removed from the feature set going forward. The honest, real result for this contract is AUC ≈ 0.59 using gsc_impressions, gsc_clicks, and ctr only

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice only covers one month (2026-03), so it can't show seasonality or long-term trend persistence — a real decline claim needs a longer window to rule out a short blip. The history is also an unbalanced panel: different clients started tracking at different times, so early rows for some clients may be GSC-only with ga4_data_available = FALSE, understating their true engagement.

This slice only covers one month (2026-03), so it can't show seasonality or long-term trend persistence — a real decline claim needs a longer window to rule out a short blip. The history is also an unbalanced panel: different clients started tracking at different times, so early rows for some clients are GSC-only, with ga4_data_available = FALSE. This is not just theoretical — Query 3 above confirms it directly: only 4.2% of rows in this March slice (413,966 of 9,841,378) have GA4 engagement data available, meaning any feature built from ga4_sessions or engagement signals would be missing for the large majority of rows


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Re-confirming the GA4 availability stat that backs the limitation above
print(f"GA4 available: {413966} of {9841378} rows ({round(413966/9841378*100, 1)}%)")

GA4 available: 413966 of 9841378 rows (4.2%)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.